In [6]:
import torch
from torch import nn

inputs = torch.tensor(
    [[0.43, 0.15, 0.89],    # Your    (x^1)  
     [0.55, 0.87, 0.66],    # Journey (x^2)
     [0.57, 0.85, 0.64],    # starts  (x^3)
     [0.22, 0.58, 0.33],    # with    (x^4)
     [0.77, 0.25, 0.10],    # one     (x^5)
     [0.05, 0.80, 0.55]]    # step    (x^6)
)
d_in = inputs.shape[1]
d_out = 2

In [7]:
class SelfAttentionV2(nn.Module):
    def __init__(self, d_in, d_out, qkv_bias=False):
        super().__init__()
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

    def forward(self, x):
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)
        d_k = keys.shape[1]
        attn_scores = queries @ keys.T
        attn_weights = torch.softmax(
            attn_scores / d_k**0.5, dim=-1
        )
        context_vec = attn_weights @ values

        return context_vec

In [8]:
torch.manual_seed(789)
sa_v2 = SelfAttentionV2(d_in, d_out)
queries = sa_v2.W_query(inputs)
keys = sa_v2.W_key(inputs)
attn_scores = queries @ keys.T
attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
print(attn_weights)

tensor([[0.1921, 0.1646, 0.1652, 0.1550, 0.1721, 0.1510],
        [0.2041, 0.1659, 0.1662, 0.1496, 0.1665, 0.1477],
        [0.2036, 0.1659, 0.1662, 0.1498, 0.1664, 0.1480],
        [0.1869, 0.1667, 0.1668, 0.1571, 0.1661, 0.1564],
        [0.1830, 0.1669, 0.1670, 0.1588, 0.1658, 0.1585],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<SoftmaxBackward0>)


In [9]:
context_length = attn_scores.shape[0]
mask_simple = torch.tril(torch.ones(context_length, context_length))
print(mask_simple)

tensor([[1., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1.]])
